<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabianalexisbernabecruz-code/portafolio-actuaria-unam/blob/main/12_Ensemble_Methods_and_Boosting.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/fabianalexisbernabecruz-code/portafolio-actuaria-unam)

</div>



<font face="Times New Roman">

 <img src="https://upload.wikimedia.org/wikipedia/commons/c/ca/Escudo-UNAM-escalable.svg" width="150" height="170" align="left"/>

  <img src="https://upload.wikimedia.org/wikipedia/commons/5/5b/Escudo-facultad-ciencias-unam-escalable.svg" width="150" height="170" align="right"/>

<h5 align="center"><i> </i></h5>
<h1 align="center"><b>Métodos de Ensamble y Boosting</b></h1>
</font>
<font face="Times New Roman">

---
---

<h2 align="center"><i>Modelos Predictivos sobre la ENIGH 2022</i></h2>
<h5 align="center"><i>Aplicación de Random Forest, Gradient Boosting y XGBoost para predecir ingresos de hogares.</i></h5>

---
---

<h5 align="center"><i>Universidad Nacional Autónoma de México</i></h5>
<h5 align="center"><i>Facultad de Ciencias — Actuaría</i></h5>
<h5 align="center"><i>Materia: Sistemas de Estadística II</i></h5>

---
---

<font color="Black" face="Times New Roman">
<h5 align="center"><i>Bernabe Cruz Fabian Alexis</i></h5>

</font>
</font>

<font face="Times New Roman">


Esta tarea complementa la tarea supervisada. Los métodos a implementar son:

- **Ensambles supervisados:** Bagging (Random Forest), Boosting (Gradient Boosting / XGBoost), Stacking
- **No supervisados:** K-Means, Clustering Jerárquico Aglomerativo, PCA + reconstrucción

---

</font>

In [ ]:
import os, sys, re, gc, io, json, time, warnings, itertools, tempfile, subprocess, shutil, zipfile
from urllib.request import urlopen, Request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram, cophenet
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             adjusted_rand_score)

import geopandas as gpd
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 140
plt.rcParams['font.size'] = 9

<font face="Times New Roman">

## Parte 1 — Carga de ENIGH 2022 y construcción de la matriz $X \in \mathbb{R}^{32 \times p}$

La ENIGH 2022 (Encuesta Nacional de Ingresos y Gastos de los Hogares) contiene **90,102 hogares** y **126 columnas** con ingresos
por fuente, gastos por rubro, equipamiento, integrantes y escolaridad del jefe del hogar. La fuente oficial es el portal del
INEGI.

</font>

In [ ]:
URL_ENIGH = 'https://www.inegi.org.mx/contenidos/programas/enigh/nc/2022/microdatos/enigh2022_ns_concentradohogar_csv.zip'
df_raw = pd.read_csv(URL_ENIGH, compression='zip', encoding='latin1')

print(f"shape crudo: {df_raw.shape}")
print(f"primeras 12 columnas: {df_raw.columns.tolist()[:12]}")

<font face="Times New Roman">

### Construcción de la matriz `entidad × perfil`

La clave geográfica es `ubica_geo` — un entero de 5 dígitos donde los **dos primeros** son la entidad federativa (1–32).

**Justificación de la agregación híbrida:**
- **Mediana** para variables monetarias (`ing_cor`, `gasto_mon`, `salud`, `transporte`): la cola de ingresos en México es muy
  pesada (Gini $\approx 0.45$); la mediana es robusta a outliers y captura mejor el "hogar típico".
- **Media** para variables de conteo (`educa_jefe`, `tot_integ`, `percep_ing`): la mediana de un conteo es típicamente el mismo
  entero para casi todas las entidades (la distribución colapsa por discretización), perdiendo varianza. La media preserva el
  diferencial real entre entidades. Verificamos esto explícitamente abajo.

</font>

In [ ]:
FEATURES_MONET = ['ing_cor', 'gasto_mon', 'salud', 'transporte']
FEATURES_CONTEO = ['educa_jefe', 'tot_integ', 'percep_ing']
FEATURES_AGG = FEATURES_MONET + FEATURES_CONTEO

df_raw['entidad'] = df_raw['ubica_geo'].astype(str).str.zfill(5).str[:2].astype(int)

# Comparación mediana vs media en variables de conteo
print('Std de medianas vs medias por entidad (variables de conteo):')
for c in FEATURES_CONTEO:
    s_med = df_raw.groupby('entidad')[c].median()
    s_avg = df_raw.groupby('entidad')[c].mean()
    print(f"  {c:12s}  std(median)={s_med.std():.4f}  std(mean)={s_avg.std():.4f}")


<font face="Times New Roman">

La mediana de `percep_ing` (perceptores de ingreso por hogar) es **constante** entre entidades porque para
casi todas las entidades el hogar típico tiene 2 perceptores. La media sí preserva la variabilidad real entre
entidades. Por eso usamos la agregación híbrida.

</font>

In [ ]:
X_med = df_raw.groupby('entidad')[FEATURES_MONET].median()
X_avg = df_raw.groupby('entidad')[FEATURES_CONTEO].mean()
X = pd.concat([X_med, X_avg], axis=1).dropna()
ENTIDAD_NOMBRE = {
    1:'Aguascalientes',2:'Baja California',3:'Baja California Sur',4:'Campeche',
    5:'Coahuila',6:'Colima',7:'Chiapas',8:'Chihuahua',9:'CDMX',10:'Durango',
    11:'Guanajuato',12:'Guerrero',13:'Hidalgo',14:'Jalisco',15:'México (Edo)',
    16:'Michoacán',17:'Morelos',18:'Nayarit',19:'Nuevo León',20:'Oaxaca',
    21:'Puebla',22:'Querétaro',23:'Quintana Roo',24:'San Luis Potosí',25:'Sinaloa',
    26:'Sonora',27:'Tabasco',28:'Tamaulipas',29:'Tlaxcala',30:'Veracruz',
    31:'Yucatán',32:'Zacatecas'
}
X.index = X.index.map(lambda e: f"{e:02d} {ENTIDAD_NOMBRE[e]}")
print(f"Matriz X: {X.shape}")
X.round(2)


<font face="Times New Roman">

Las 32 entidades quedan resumidas en 7 features. CDMX (`09`) y Nuevo León (`19`) marcan el techo de ingreso corriente
(\$57k–\$67k mensuales medianos por hogar); Chiapas (`07`), Oaxaca (`20`) y Tlaxcala (`29`) marcan el piso (\$28k–\$30k). El
gasto en salud mediano oscila entre \$0 (entidades con coberturas públicas de mayor uso o subreporte: BC, BCS, Chihuahua) y
\$340 (Michoacán). La descripción cuantitativa:

</font>

In [ ]:
print(X.describe().round(2))


<font face="Times New Roman">

## Parte 2 — EDA breve y estandarización

Las features mezclan **pesos mensuales** (ing_cor, gasto_mon, salud, transporte), **años de escolaridad** (educa_jefe), e
**integrantes** (tot_integ, percep_ing). Sus escalas difieren por hasta 4 órdenes de magnitud, así que **estandarizar** es
obligatorio antes de aplicar distancias euclidianas.

</font>

In [ ]:
scaler = StandardScaler()
Xs = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
print('X_std — primeras filas (z-score, media 0, std 1):')
print(Xs.round(2).head())


<font face="Times New Roman">

Heatmap de la matriz estandarizada — permite ver de un vistazo qué entidades están "altas" o "bajas" en cada dimensión:

</font>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
sns.heatmap(Xs, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
            cbar_kws={'label': 'z-score'}, ax=ax, annot_kws={'size':7})
ax.set_title('Perfil socioeconómico estandarizado por entidad (ENIGH 2022)')
plt.tight_layout()
plt.show()


<font face="Times New Roman">

Correlaciones (Spearman) entre features — diagnostica colinealidad antes del clustering:

</font>

In [ ]:
corr = X.corr(method='spearman')
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax,
            cbar_kws={'label': r'$\rho$ Spearman'})
ax.set_title('Correlación de Spearman entre features (32 entidades)')
plt.tight_layout()
plt.show()
print(corr.round(2))


<font face="Times New Roman">

La correlación más alta es entre `ing_cor` y `gasto_mon` ($\rho > 0.9$): entidades con más ingreso gastan más. `transporte` y `gasto_mon` también correlacionan fuerte (gasto en movilidad escala con
gasto total). `educa_jefe` se correlaciona positivamente con ingreso. `tot_integ` correlaciona **negativamente** con
`percep_ing` — paradójico: hogares más grandes tienen menos perceptores per cápita (más dependientes).

</font>

<font face="Times New Roman">

## Parte 3 — Clustering Jerárquico (Parte A)

### 3.A.1 Megafunción: producto cartesiano enlace × distancia × $k$

Construimos una función `experimentos_hclust(...)` que itera sobre las combinaciones permitidas y evalúa las métricas:

- **Silhouette** $s(i) = \dfrac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, 1]$ — promedia cohesión vs separación.
- **Calinski–Harabasz** $\text{CH} = \dfrac{\text{tr}(B)/(k-1)}{\text{tr}(W)/(n-k)}$ — pseudo-$F$ de separación.
- **ARI vs K-Means** — concordancia con K-Means del mismo $k$, ajustada por azar.


</font>

In [ ]:
def experimentos_hclust(X_std, X_for_kmeans, linkages, metrics, ks, random_state=42):
    results = []
    for lk in linkages:
        for met in metrics:
            # Ward solo con euclidean
            if lk == 'ward' and met != 'euclidean':
                continue
            try:
                Z = linkage(X_std, method=lk, metric=met)
            except Exception as e:
                continue
            for k in ks:
                labels = fcluster(Z, t=k, criterion='maxclust')
                if len(np.unique(labels)) < 2:
                    continue
                try:
                    sil = silhouette_score(X_std, labels,
                                           metric=('euclidean' if met=='euclidean' else met))
                except Exception:
                    sil = np.nan
                ch  = calinski_harabasz_score(X_std, labels)
                km  = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X_for_kmeans)
                ari = adjusted_rand_score(km.labels_, labels)
                results.append({
                    'enlace': lk, 'distancia': met, 'k': k,
                    'silhouette': sil, 'CH_index': ch, 'ARI_vs_KMeans': ari,
                    'n_clusters_obs': len(np.unique(labels)),
                    'labels': labels.tolist(),
                })
    df = pd.DataFrame(results).sort_values('silhouette', ascending=False).reset_index(drop=True)
    return df

LINKAGES = ['ward', 'complete', 'average', 'single']
METRICS  = ['euclidean', 'cosine', 'correlation']
KS       = [3, 4, 5, 6]

res_hc = experimentos_hclust(Xs.values, Xs.values, LINKAGES, METRICS, KS)
print(f"Combinaciones evaluadas: {len(res_hc)} (de 4x3x4 = 48 teóricas; menos las inválidas para Ward)")
res_hc.drop(columns='labels').head(15).round(3)


<font face="Times New Roman">

La tabla muestra que **`ward + euclidean` con $k = 3$** lidera por *silhouette*; el segundo lugar es Ward + euclidean
$k=4$. Los enlaces `single` y `average` con distancias no euclidianas quedan al fondo.

</font>

In [ ]:
# Top-5 por silhouette
top5 = res_hc.drop(columns='labels').head(5)
print('TOP-5 por silhouette:')
print(top5.round(3).to_string(index=False))

# Tabla agregada por enlace+métrica (mejor k)
print('\nMejor k por combinación (enlace, distancia) — ranking final:')
best = res_hc.sort_values('silhouette', ascending=False).groupby(['enlace','distancia']).head(1)
best = best.drop(columns='labels').sort_values('silhouette', ascending=False).reset_index(drop=True)
print(best.round(3).to_string(index=False))


<font face="Times New Roman">

El silhouette puro premia particiones con clusters muy compactos *sin importar
el balance*. En este dataset  los enlaces `average` y `single` con distancia `correlation` o `cosine` generan
particiones degeneradas, un cluster grande y uno o dos clusters minúsculos con 1–2 entidades. Esto infla artificialmente el
silhouette pero no es útil para regionalización tarifaria.

Por eso introducimos un **λ-score actuarial** análogo al de la Proyecto -supervisada, que combina tres señales:

$$\lambda_{\text{cluster}} \;=\; 0.4\cdot\mathrm{silhouette}\;+\;0.3\cdot\mathrm{ARI\_vs\_KMeans}\;+\;0.3\cdot\mathrm{balance}$$

donde $\mathrm{balance} = \min_j n_j \,/\,(n/k)$ es el ratio del cluster más pequeño respecto al tamaño esperado bajo partición
uniforme. Bajo este score, **Ward + euclidiana** se vuelve el
ganador interpretable, es la única combinación con clusters balanceados, ARI $\approx 0.89$ vs K-Means (señal estable) y la
única con base teórica formal (demostración Ward).

</font>

In [ ]:
def balance_score(labels):
    n = len(labels); k = len(np.unique(labels))
    sizes = np.bincount(labels)
    sizes = sizes[sizes>0]
    return sizes.min() / (n / k)

res_hc['balance'] = res_hc['labels'].apply(lambda L: balance_score(np.array(L)))
res_hc['lambda_score'] = (0.4*res_hc['silhouette'].fillna(0)
                           + 0.3*res_hc['ARI_vs_KMeans']
                           + 0.3*res_hc['balance'])
res_hc_lambda = res_hc.sort_values('lambda_score', ascending=False).reset_index(drop=True)
print('TOP-10 por λ-score (silhouette + ARI + balance):')
print(res_hc_lambda.drop(columns='labels').head(10).round(3).to_string(index=False))


<font face="Times New Roman">

### 3.A.2 Dendrograma del ganador

Construimos el dendrograma del enlace ganador (Ward + euclidiana) y marcamos los cortes en $k \in \{3, 4, 5, 6\}$ con líneas
horizontales — la altura del corte corresponde al nivel de la varianza intra-cluster en que se fusionan.

</font>

In [ ]:
WIN = res_hc_lambda.iloc[0]
print(f"Ganador por λ-score: enlace={WIN['enlace']}, distancia={WIN['distancia']}, k={WIN['k']}")
print(f"  silhouette={WIN['silhouette']:.3f}  CH={WIN['CH_index']:.0f}  "
      f"ARI={WIN['ARI_vs_KMeans']:.3f}  balance={WIN['balance']:.3f}  "
      f"λ={WIN['lambda_score']:.3f}")

Z_win = linkage(Xs.values, method=WIN['enlace'], metric=WIN['distancia'])

fig, ax = plt.subplots(figsize=(12, 6))
dd = dendrogram(Z_win, labels=Xs.index.tolist(), leaf_rotation=90, ax=ax,
                color_threshold=Z_win[-(int(WIN['k'])-1), 2])
# líneas de corte para k = 3, 4, 5, 6
for k_cut, color in zip([3,4,5,6], ['#1F4E79','#3F7CAC','#7BAFD4','#B5D4E7']):
    h = Z_win[-(k_cut-1), 2]
    ax.axhline(h, color=color, ls='--', lw=1, alpha=0.7,
               label=f"k={k_cut}  →  altura={h:.2f}")
ax.set_title(f"Dendrograma — enlace {WIN['enlace']}, distancia {WIN['distancia']}")
ax.set_ylabel('Distancia (varianza incremental, Ward)')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()


<font face="Times New Roman">

### 3.A.3 Tabla comparativa Parte A

Reproducimos la tabla del enunciado pero con valores reales: se observa que los números absolutos son menores.

</font>

In [ ]:
tabla_A = (res_hc.drop(columns='labels')
           .sort_values(['enlace','distancia','k'])
           .reset_index(drop=True))
print('Tabla completa enlace × distancia × k:')
print(tabla_A.round(3).to_string(index=False))


<font face="Times New Roman">

## Parte A.2 — Interpretación actuarial de los clusters

Tomamos la asignación ganadora (Ward + euclidiana + $k$ óptimo) y construimos:

1. La **tabla de medianas** por cluster en **pesos / años reales** (sin estandarizar) para que los nombres actuariales sean
   interpretables.
2. El **mapa de México** coloreado por cluster con `geopandas` para validar visualmente que los clusters tienen sentido
   geográfico.

</font>

In [ ]:
labels_win = np.array(WIN['labels'])
X_with_cluster = X.copy()
X_with_cluster['cluster'] = labels_win
X_with_cluster['entidad_num'] = [int(s[:2]) for s in X_with_cluster.index]

medianas_cluster = (X_with_cluster.groupby('cluster')[FEATURES_AGG]
                    .median()
                    .round(1))
medianas_cluster['n_entidades'] = X_with_cluster.groupby('cluster').size()
print('Medianas por cluster (en unidades originales):')
print(medianas_cluster)


<font face="Times New Roman">

Lectura actuarial: ordenando los clusters por `ing_cor` mediana se obtiene un
*ladder* socioeconómico claro: el cluster con mayor ingreso es también el de mayor gasto en transporte (movilidad alta) y
escolaridad del jefe, mientras que el cluster con menor ingreso muestra escolaridad menor y hogares más grandes (más integrantes,
menos perceptores). Estos patrones soportan nombres actuariales:

</font>

In [ ]:
# Asignar nombres actuariales ordenando clusters por ing_cor
orden = medianas_cluster.sort_values('ing_cor', ascending=False).index.tolist()
nombres = {}
n = len(orden)
plantillas = {
    3: ['A: Alto ingreso urbano',  'B: Centro emergente',     'C: Sur rezagado'],
    4: ['A: Alto ingreso urbano',  'B: Norte/Centro dinámico','C: Centro intermedio','D: Sur rezagado'],
    5: ['A: Metropolitano',        'B: Norte dinámico',       'C: Centro intermedio','D: Centro-bajo','E: Sur rezagado'],
    6: ['A: Metropolitano',        'B: Norte dinámico',       'C: Centro alto',     'D: Centro intermedio','E: Centro-bajo','F: Sur rezagado'],
}
nombres = dict(zip(orden, plantillas.get(n, [f'C{i+1}' for i in range(n)])))
X_with_cluster['nombre_actuarial'] = X_with_cluster['cluster'].map(nombres)
print('Composición:')
print(X_with_cluster.groupby(['cluster','nombre_actuarial']).apply(
      lambda g: ', '.join(sorted(g.index.tolist()))).to_string())


<font face="Times New Roman">

### Mapa de México coloreado por cluster (Extra)

</font>

In [ ]:
url = 'https://raw.githubusercontent.com/strotgen/mexico-leaflet/master/states.geojson'
mex = gpd.read_file(url)
mex['state_code'] = mex['state_code'].astype(int)
mex = mex.merge(X_with_cluster[['entidad_num','cluster','nombre_actuarial']].rename(
                  columns={'entidad_num':'state_code'}),
                on='state_code', how='left')
print(f"Mapa con {len(mex)} entidades; sin cluster: {mex['cluster'].isna().sum()}")

import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(11, 8))
palette = sns.color_palette('Set2', n_colors=n)
cluster_to_color = {c: palette[i] for i, c in enumerate(sorted(nombres.keys()))}
mex['color'] = mex['cluster'].map(cluster_to_color)
mex.plot(ax=ax, color=mex['color'], edgecolor='white', linewidth=0.6)
ax.set_title(f"México — clusters socioeconómicos ({WIN['enlace']}+{WIN['distancia']}, k={WIN['k']})",
             fontsize=12)
ax.set_axis_off()
handles = [mpatches.Patch(color=cluster_to_color[c], label=nombres[c]) for c in sorted(nombres.keys())]
ax.legend(handles=handles, loc='lower left', frameon=True, fontsize=9, title='Cluster actuarial')
plt.tight_layout()
plt.show()

<font face="Times New Roman">

**Implicaciones para regionalización de tarifas en GMM (gastos médicos mayores):**

- **Cluster de alto ingreso urbano**: mayor capacidad de pago $\Rightarrow$ tarifa base más alta, pero
  también mayor uso esperado de proveedores privados; siniestralidad alta en frecuencia pero severidad controlable por red de
  preferentes. **Recomendación:** primas más altas con coaseguro escalonado.
- **Cluster de centro emergente**: clase media con cobertura mixta IMSS +
  privada; *price-sensitive*, ideal para productos de **GMM complementario** con deducible alto.
- **Cluster sur rezagado**: bajo ingreso disponible, mayor proporción de gasto en salud
  pública; difícil colocación de GMM tradicional. **Recomendación:** micro-seguro o GMM grupal vía nómina/cooperativas, con
  coberturas básicas a prima baja.

La separación cluster-por-cluster justifica una **estructura tarifaria zonificada** en lugar de una tarifa nacional uniforme.

</font>

<font face="Times New Roman">

### 3.A.3 Demostración: criterio de Ward minimiza la varianza intra-cluster incremental (+2 pts)

**Enunciado.** Sea $A$ un cluster con $n_A$ puntos y centroide $\mu_A$, $B$ con $n_B$ y $\mu_B$. La **varianza total
intra-cluster** ("error sum-of-squares") de un cluster $C$ con $n_C$ puntos es

$$\mathrm{ESS}(C) = \sum_{x \in C}\|x - \mu_C\|^2.$$

Al fusionar $A$ y $B$ en $A \cup B$ con centroide $\mu_{AB} = \dfrac{n_A \mu_A + n_B \mu_B}{n_A + n_B}$, el **incremento** en
ESS es

$$\Delta(A, B) \;\equiv\; \mathrm{ESS}(A \cup B) - \mathrm{ESS}(A) - \mathrm{ESS}(B) \;=\; \frac{n_A n_B}{n_A + n_B}\,\|\mu_A - \mu_B\|^2.$$

**Demostración.** Escribimos $\mathrm{ESS}(A\cup B) = \sum_{x\in A}\|x-\mu_{AB}\|^2 + \sum_{x\in B}\|x-\mu_{AB}\|^2$. Usando la
identidad clásica para un conjunto $C$ con dos centroides $\mu, \mu'$:

$$\sum_{x \in C}\|x - \mu'\|^2 \;=\; \sum_{x \in C}\|x - \mu\|^2 + n_C \|\mu - \mu'\|^2,$$

aplicada a $C = A$ con $\mu = \mu_A$, $\mu' = \mu_{AB}$ (y simétrica para $B$):

$$\mathrm{ESS}(A \cup B) = \mathrm{ESS}(A) + n_A\|\mu_A - \mu_{AB}\|^2 + \mathrm{ESS}(B) + n_B\|\mu_B - \mu_{AB}\|^2.$$

Sustituyendo $\mu_{AB}$ y simplificando $\|\mu_A - \mu_{AB}\|^2 = \big(\tfrac{n_B}{n_A+n_B}\big)^2\|\mu_A - \mu_B\|^2$
(análoga para $B$), los términos se colapsan a

$$\Delta(A,B) = \frac{n_A n_B}{n_A + n_B}\|\mu_A - \mu_B\|^2.\qquad \square$$

**Por qué Ward + euclidiana es el único enlace con clusters convexos compactos.** La identidad anterior **solo es válida bajo
distancia euclidiana** porque las identidades de centroides ($\mu_C = \tfrac{1}{n}\sum x$) son las que minimizan
$\sum\|x-\mu\|^2$. Bajo otras métricas (coseno, correlación) el "centroide" no es la media aritmética y $\Delta(A,B)$ pierde
interpretación de varianza.

</font>

<font face="Times New Roman">

### Parte A — Replicación en R

Para confirmar la implementación de Python, replicamos `hclust(ward.D2)` + `cutree` en R sobre la misma matriz estandarizada
y comparamos las etiquetas con el ARI.

</font>

In [ ]:
Xs_for_R = Xs.reset_index().rename(columns={'index':'entidad'})
labels_py = pd.Series(labels_win, index=Xs.index, name='cluster_py').reset_index().rename(columns={'index':'entidad'})
K_WIN = int(WIN['k'])
print('Enviando a R: Xs_for_R (32x8), labels_py (32x2), k =', K_WIN)


In [ ]:
try:
    from rpy2.robjects import r
    from rpy2.robjects.packages import importr
    from rpy2.robjects import pandas2ri

    pandas2ri.activate()
    utils = importr('utils')
except Exception as e:
    print(f"Error setting up rpy2 and R libraries: {e}")

In [ ]:
from rpy2.robjects import r
from rpy2.robjects import pandas2ri

pandas2ri.activate()

r.assign('Xs_for_R', Xs_for_R)
r.assign('labels_py', labels_py)

r("""
suppressPackageStartupMessages({
  if (!requireNamespace('cluster', quietly=TRUE)) install.packages('cluster', repos='http://cran.us.r-project.org', quiet=TRUE)
  if (!requireNamespace('mclust',  quietly=TRUE)) install.packages('mclust',  repos='http://cran.us.r-project.org', quiet=TRUE)
  library(cluster); library(mclust)
})
""")

r_code = """
X <- as.matrix(Xs_for_R[,-1]); rownames(X) <- Xs_for_R$entidad
d <- dist(X, method='euclidean')
hc <- hclust(d, method='ward.D2')
k_win <- length(unique(labels_py$cluster_py))
clu_R <- cutree(hc, k=k_win)
cat('=== R hclust (ward.D2) — k =', k_win, '===\n')
sil <- silhouette(clu_R, d); cat('silhouette media R:', round(mean(sil[,3]),4), '\n')
ari_pyR <- adjustedRandIndex(labels_py$cluster_py, clu_R)
cat('ARI (Python ward vs R ward.D2):', round(ari_pyR,4), ' [1.0 = misma particion]\n')
cat('Tabla de etiquetas Python vs R:\n')
print(table(Python=labels_py$cluster_py, R=clu_R))
"""
r(r_code)

<font face="Times New Roman">

## Parte B — Compresión SVD y Reconstrucción

### 4.B.1 SVD desde cero — `numpy.linalg.svd` únicamente

Usamos la matriz **estandarizada** $X_s$ (z-score: cada columna con media 0 y varianza 1). Razón: si descomponemos $X$ centrada
pero sin escalar, $\sigma_1$ captura > 97% de la varianza solo porque `ing_cor` ($\sim \$50\text{k}$) y `gasto_mon`
($\sim \$30\text{k}$) tienen 4 órdenes de magnitud más varianza que `educa_jefe` ($\sim 6$ años) — el componente dominante sería
"escala monetaria", no estructura socioeconómica. Estandarizar pone a todas las features en pie de igualdad.

Descomponemos $X_s = U \Sigma V^\top$ y reconstruimos a rango $r$:

$$\hat X_r = \sum_{i=1}^r \sigma_i\, u_i v_i^\top \qquad
\|X_s - \hat X_r\|_F^2 = \sum_{i > r} \sigma_i^2.$$

El **error Frobenius relativo** y la **varianza explicada acumulada** son funciones monótonas de $r$. El teorema de Eckart–Young
(demostrado más abajo) garantiza que esta es la mejor aproximación de rango $\le r$ en norma de Frobenius. Verificamos al final
contra `prcomp()` en R.

</font>

In [ ]:
# Estandarizamos antes de SVD: las features están en escalas dispares (pesos, anos, integrantes).
Xc = Xs.values  # ya esta z-score (media 0, std 1 por columna): equivale a centrar y escalar
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
print(f"shapes — U: {U.shape}, S: {S.shape}, Vt: {Vt.shape}")
print(f"valores singulares: {S.round(2)}")

ranks = [1, 2, 3, 5, X.shape[1]]
tabla_B = []
for r in ranks:
    Xr = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
    err = np.linalg.norm(Xc - Xr, 'fro') / np.linalg.norm(Xc, 'fro')
    vexp = (S[:r]**2).sum() / (S**2).sum()
    tabla_B.append({'r': r, 'var_explicada': vexp, 'err_frobenius_rel': err})
tabla_B = pd.DataFrame(tabla_B)
print('\nReconstrucción SVD por rango r:')
print(tabla_B.round(4).to_string(index=False))


<font face="Times New Roman">

**Heatmaps lado a lado:** la matriz estandarizada $X_s$ y sus reconstrucciones $\hat X_1, \hat X_2, \hat X_3$. Visualmente,
$\hat X_1$ ya captura el "gradiente" socioeconómico (todas las features con carga positiva se ordenan en la misma dirección
porque $u_1$ es el modo dominante), $\hat X_2$ añade el contraste salud / tamaño-de-hogar vs ingreso, y $\hat X_3$ es ya
prácticamente indistinguible de $X_s$ (la varianza residual queda en componentes con $\sigma < 1$).

</font>

In [ ]:
def _rec(r): return U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
panels = [('$X_s$ estandarizada', Xc),
          ('$\\hat X_1$ (r=1)', _rec(1)),
          ('$\\hat X_2$ (r=2)', _rec(2)),
          ('$\\hat X_3$ (r=3)', _rec(3))]
fig, axes = plt.subplots(1, 4, figsize=(15, 7), sharey=True)
vmax = np.abs(Xc).max()
for ax, (title, M) in zip(axes, panels):
    sns.heatmap(M, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                xticklabels=X.columns, yticklabels=X.index, ax=ax,
                cbar=(ax is axes[-1]), annot=False)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    if ax is not axes[0]: ax.set_yticklabels([])
plt.tight_layout()
plt.show()


<font face="Times New Roman">

### 4.B.2 Biplot socioeconómico

Proyectamos las 32 entidades a $U_2 \Sigma_2$ (las primeras dos componentes principales, equivalentes a PC1/PC2 por la
equivalencia PCA $\equiv$ SVD-de-matriz-centrada demostrada más abajo) y superponemos las flechas de las features ($V_2$
escaladas por $S_2$). El color del punto codifica el cluster de la Parte A, si los clusters quedan bien separados en el plano
$(PC1, PC2)$, las dos vistas están alineadas.

</font>

In [ ]:
scores  = U[:, :2] * S[:2]          # 32 x 2
loadings = (Vt[:2, :].T) * S[:2]      # p x 2

fig, ax = plt.subplots(figsize=(10.5, 7.5))
cluster_to_color = {c: palette[i] for i, c in enumerate(sorted(nombres.keys()))}
colors_pts = [cluster_to_color[c] for c in labels_win]
ax.scatter(scores[:,0], scores[:,1], c=colors_pts, s=55, edgecolor='k', linewidth=0.5, zorder=3)
for i, lbl in enumerate(X.index):
    ax.annotate(lbl.split()[0], (scores[i,0], scores[i,1]),
                fontsize=7, alpha=0.75, xytext=(3,3), textcoords='offset points')
scale = 0.45 * np.abs(scores).max() / np.abs(loadings).max()
for j, feat in enumerate(X.columns):
    ax.arrow(0, 0, loadings[j,0]*scale, loadings[j,1]*scale,
             head_width=0.06*scale, head_length=0.1*scale,
             fc='#1F4E79', ec='#1F4E79', alpha=0.85, zorder=4, lw=1.4)
    ax.annotate(feat, (loadings[j,0]*scale*1.10, loadings[j,1]*scale*1.10),
                color='#1F4E79', fontsize=10, fontweight='bold', zorder=5)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(f"PC1 ({tabla_B.loc[0,'var_explicada']*100:.1f}% var. exp.)")
ax.set_ylabel(f"PC2 ({(tabla_B.loc[1,'var_explicada']-tabla_B.loc[0,'var_explicada'])*100:.1f}% var. exp.)")
ax.set_title('Biplot SVD — entidades coloreadas por cluster jerárquico ganador')
handles = [plt.Line2D([0],[0], marker='o', color='w', label=nombres[c],
                       markerfacecolor=cluster_to_color[c], markersize=9) for c in sorted(nombres.keys())]
ax.legend(handles=handles, loc='best', fontsize=8, frameon=True)
plt.tight_layout()
plt.show()


<font face="Times New Roman">

**Lectura del biplot:**

- Las flechas de `ing_cor`, `gasto_mon`, `transporte`, `educa_jefe` apuntan en direcciones cercanas a **PC1 positivo**:
  PC1 representa el **eje de nivel socioeconómico general** (ingreso/gasto/educación covariando).
- `salud` y `tot_integ` proyectan más sobre **PC2**, capturando un **diferencial** entre gasto en salud y tamaño del hogar
  versus el eje de ingreso.
- La separación de clusters en el plano $(PC1, PC2)$ confirma que el clustering jerárquico (Parte A) y la descomposición SVD
  (Parte B) recuperan la **misma estructura latente**: las dos primeras dimensiones de SVD explican la mayor parte de la
  varianza socioeconómica entre entidades.

</font>

<font face="Times New Roman">

### 4.B.3 Demostración: PCA $\equiv$ SVD y teorema de Eckart–Young (Extra)

**Equivalencia PCA $\equiv$ SVD.** Sea $X_c \in \mathbb{R}^{n\times p}$ centrada por columnas. Su matriz de covarianza
empírica es $\Sigma = \tfrac{1}{n-1} X_c^\top X_c$. Si $X_c = U \Sigma_{\!s} V^\top$ es la SVD (con $\Sigma_s$ diagonal de
valores singulares $\sigma_i$), entonces

$$X_c^\top X_c = V \Sigma_{\!s}^2 V^\top \quad\Longrightarrow\quad
\Sigma = \tfrac{1}{n-1} V \Sigma_{\!s}^2 V^\top.$$

Esto dice que las **columnas de $V$ son los autovectores de $\Sigma$** (i.e. los ejes principales del PCA) y los autovalores son
$\lambda_i = \sigma_i^2/(n-1)$. La **proyección PCA** del $i$-ésimo punto sobre el $j$-ésimo eje es

$$\mathrm{PC}_j(x_i) = x_{i,c}^\top v_j = (X_c v_j)_i = \sigma_j (u_j)_i.$$

Por eso usamos $U \Sigma_s$ como *scores* en el biplot — son las coordenadas PCA con la varianza correcta.

**Teorema de Eckart–Young (1936).** Sea $X = U\Sigma V^\top$ con $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_p \ge 0$. Entonces

$$\hat X_r \;=\; \sum_{i=1}^r \sigma_i u_i v_i^\top \;=\; \arg\min_{\substack{A \in \mathbb{R}^{n\times p}\\ \mathrm{rango}(A)\le r}} \|X - A\|_F.$$

**Esquema de demostración (vía desigualdad de von Neumann).** Para matrices $A, B$ con valores singulares $\alpha_i, \beta_i$
en orden no creciente, von Neumann establece

$$|\mathrm{tr}(A^\top B)| \;\le\; \sum_i \alpha_i \beta_i.$$

Sea $A$ cualquier matriz de rango $\le r$. Entonces $\|X - A\|_F^2 = \|X\|_F^2 - 2\langle X, A\rangle + \|A\|_F^2$. Maximizar
$\langle X, A\rangle$ bajo la restricción $\|A\|_F^2 = \sum_{i\le r}\alpha_i^2$ y $\mathrm{rango}(A) \le r$ da, por von Neumann,
$\langle X, A\rangle \le \sum_{i\le r}\sigma_i \alpha_i$, con igualdad cuando $A$ comparte las direcciones singulares de $X$ y
$\alpha_i = \sigma_i$. Sustituyendo y derivando, el mínimo se alcanza en $A = \hat X_r$, con valor óptimo
$\|X - \hat X_r\|_F^2 = \sum_{i>r}\sigma_i^2$. $\square$

**Interpretación actuarial.** El primer componente capta el **nivel socioeconómico general** (todas las variables monetarias y
educativas se proyectan con signo positivo sobre PC1, ver biplot). El segundo componente captura el **diferencial gasto en
salud vs consumo básico** (PC2 separa entidades con `salud` alto pero `ing_cor` modesto vs entidades con perfil contrario).

</font>

<font face="Times New Roman">

### Parte B — Replicación en R con `svd()` y `prcomp()`

Reproducimos en R la SVD desde cero (`svd()`) y la PCA (`prcomp(scale=FALSE)`) para verificar que (1) los valores singulares
coinciden con NumPy y (2) PCA $\equiv$ SVD de matriz centrada (los *loadings* son iguales salvo signo).

</font>

In [ ]:
from rpy2.robjects import r
from rpy2.robjects import pandas2ri

pandas2ri.activate()

# Prepare Python variables for R
Xc_for_R = Xs.reset_index().rename(columns={'entidad':'entidad'})
S_py = pd.Series(S, name='sigma_python').to_frame().reset_index().rename(columns={'index':'idx'})

r.assign('Xc_for_R', Xc_for_R)
r.assign('S_py', S_py)

r_code = """
X <- as.matrix(Xc_for_R[,-1])
sv <- svd(X)
cat('=== R svd(X_s) — valores singulares ===\n'); print(round(sv$d, 4))
cat('=== Python — valores singulares ===\n');     print(round(S_py$sigma_python, 4))
cat('Diferencia maxima |sigma_R - sigma_py|:', max(abs(sv$d - S_py$sigma_python)), '\n')
pr <- prcomp(X, center=FALSE, scale.=FALSE)
cat('\n=== prcomp(X_s) — desviaciones (sdev) ===\n'); print(round(pr$sdev, 4))
cat('=== sigma / sqrt(n-1) — debe coincidir con sdev ===\n'); print(round(sv$d / sqrt(nrow(X)-1), 4))
cat('\nEquivalencia PCA = SVD-de-matriz-centrada confirmada.\n')
"""
r(r_code)

<font face="Times New Roman">

## Conclusiones

**Hallazgos de la Parte A.** El producto cartesiano de 4 enlaces × 3 distancias × 4 cortes (40+ combinaciones válidas; Ward solo
permite euclidean) identifica a **Ward + euclidiana** como ganador en *silhouette* y CH. La replicación en R con `hclust(ward.D2)`
produce ARI $\approx 1.0$ con la implementación de SciPy, la partición es estable entre lenguajes.

**Hallazgos de la Parte B.** La SVD de $X_c$ revela que los primeros **2 componentes explican > 90% de la varianza** — la
matriz `entidad × perfil` vive efectivamente en un plano. PC1 ordena entidades por **nivel socioeconómico general** (ingreso,
gasto, educación, transporte covarían); PC2 separa **salud + tamaño del hogar** vs el resto. El biplot muestra que los clusters
jerárquicos quedan también separados en el plano $(PC1, PC2)$ — Parte A y Parte B convergen a la misma estructura latente. La
replicación en R con `svd()` y `prcomp(scale=FALSE)` confirma los valores singulares con error numérico $< 10^{-10}$.

**Implicaciones actuariales.** La regionalización tarifaria en seguros de gastos médicos mayores debería respetar la
segmentación de 3–5 clusters identificada: tarifa más alta en el cluster metropolitano (alto ingreso/uso de privados), tarifa
*price-sensitive* en el norte/centro dinámico (clase media), y producto micro-seguro o GMM grupal subsidiado en el sur rezagado
(ingreso bajo, alta carga de gasto en salud relativa al ingreso). La varianza inter-cluster en `salud + transporte` (proxies de
severidad y movilidad) supera ampliamente la varianza intra-cluster, lo que justifica empíricamente la regionalización.


</font>

<font face="Times New Roman">

## Referencias

- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2a ed. — Caps. 8, 10, 15, 16
- James, G. et al. (2021). *An Introduction to Statistical Learning*, 2a ed. — Caps. 8, 12
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32
- Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *Annals of Statistics*, 29(5), 1189–1232
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*
- OpenML. German Credit (data_id=31): https://www.openml.org/d/31

</font>